# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 15: Análise de Ortogonalidade, Efeito Oráculo e Grau de Consenso no MLP — Classe MEDIUM

**Autor:** Evandro Rocha (UFRJ / COPPE / PEE)
**Data:** Outubro de 2026

---

## Resumo Executivo da Investigação com Redes Neurais (MLP)

A classe **`MEDIUM`** (rebocadores e navios de apoio offshore de $50\text{ m}$ a $100\text{ m}$) foi avaliada em todas as **1.490 instâncias de teste** nos 10 folds oficiais do protocolo $5\times2$cv com a arquitetura **MLP PyTorch `[32, 16]`** nas 4 representações (**MEL, LOFAR, HHT e DEMON**).

O objetivo é confrontar os resultados do classificador SVM (Notebook 12) com os da rede neural profunda para verificar se o comportamento de consenso e o teto do oráculo se mantêm estáveis entre famílias distintas de aprendizado de máquina.

### Principais Descobertas no MLP:
1. **Teto Teórico do Oráculo MLP Atinge $84{,}90\%$ (1.265 / 1.490 áudios):**
   - O Oráculo do MLP em `MEDIUM` supera até mesmo o do SVM ($83{,}69\%$), proporcionando um ganho de **$+20{,}8$ pontos percentuais** sobre o melhor modelo isolado (MEL com $64{,}09\%$).
   - O erro comum irrecuperável é de apenas **$15{,}10\%$ (225 áudios)**.
2. **Confirmação do Alto Consenso em MEDIUM ($29{,}93\%$):**
   - Quase **$30\%$ de todos os rebocadores (446 áudios)** são acertados de forma unânime pelas 4 representações, quase $6\times$ maior que a taxa de unanimidade de `SMALL` ($5{,}34\%$ no MLP).
3. **HHT e MEL Lideram os Resgates Exclusivos no MLP:**
   - O MEL salva sozinho $88$ rebocadores ($5{,}91\%$) e o HHT salva $78$ ($5{,}23\%$).
4. **A Confusão Simétrica `MEDIUM` $\rightarrow$ `SMALL` Persiste no MLP:**
   - No MEL, **$56{,}45\%$ dos erros de `MEDIUM` vão para `SMALL`**; no LOFAR, **$58{,}01\%$ dos erros vão para `SMALL`**.
5. **A Votação por Maioria Simples ($\\ge 2/4$) Supera o Melhor Modelo Isolado:**
   - Um ensemble cego por maioria simples atinge **$65{,}84\%$ ($981$ áudios)**, superando o MEL isolado ($64{,}09\%$) em **$+1{,}75$ p.p.**.

In [ ]:
# 1. Configuração do Ambiente e Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

# Estilização publication-ready (sem dependência externa)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 15,
    'figure.dpi': 150
})

print("Ambiente configurado com sucesso! Bibliotecas prontas.")

## 2. Metodologia Experimental e Base de Dados de Teste

Avaliação sobre todos os $1.490$ testes onde a classe real comprovada era **`MEDIUM` (Classe 1)** sob o protocolo oficial $5\times2$cv com modelos **MLP PyTorch `[32, 16]`**:


In [ ]:
# 2. Dados Consolidados das Predições do MLP na Classe MEDIUM
total_med = 1490

# Desempenho individual no MLP
hits_data = {
    'Representacao': ['DEMON (253-D)', 'HHT Puro (256-D)', 'LOFAR (1024-D)', 'MEL (256-D)', 'ORÁCULO MLP (União 4 Modos)'],
    'Acertos': [686, 863, 916, 955, 1265],
    'Recall_Pct': [46.04, 57.92, 61.48, 64.09, 84.90]
}
df_hits = pd.DataFrame(hits_data)

# Grau de consenso no MLP
consensus_data = {
    'Nivel_Consenso': [
        '4 de 4 (Unanimidade / Independente)',
        '3 de 4 (Alto Consenso)',
        '2 de 4 (Consenso Parcial / Dividido)',
        '1 de 4 (Frágil / Resgate Exclusivo)',
        '0 de 4 (Erro Comum / Incorrigível)'
    ],
    'Audios': [446, 282, 253, 284, 225],
    'Pct_Total_Medium': [29.93, 18.93, 16.98, 19.06, 15.10],
    'Pct_Dentro_Oraculo': [35.26, 22.29, 20.00, 22.45, np.nan]
}
df_consensus = pd.DataFrame(consensus_data)

# Resgates exclusivos por representação no MLP
exclusive_data = {
    'Representacao': ['LOFAR Exclusivo', 'DEMON Exclusivo', 'HHT Exclusivo', 'MEL Exclusivo'],
    'Audios_Salvos': [51, 67, 78, 88],
    'Percentual': [3.42, 4.50, 5.23, 5.91]
}
df_exclusive = pd.DataFrame(exclusive_data)

# Padrão de confusão quando o MLP erra em MEDIUM
confusion_pattern_data = {
    'Representacao': ['MEL (535 erros)', 'LOFAR (574 erros)', 'HHT (627 erros)', 'DEMON (804 erros)'],
    'Confundido_SMALL_Pct': [56.45, 58.01, 38.28, 42.04],
    'Confundido_LARGE_Pct': [36.64, 35.89, 31.74, 28.11],
    'Confundido_AMBIENTAL_Pct': [6.92, 6.10, 29.98, 29.85]
}
df_confusion_pattern = pd.DataFrame(confusion_pattern_data)

display(df_hits)
display(df_consensus)

## 3. O Efeito Oráculo no MLP: Recall Individual vs. Teto Teórico

O classificador MLP atinge **$64{,}09\%$ de recall em `MEDIUM`** com MEL e **$61{,}48\%$** com LOFAR.

Contudo, a combinação das 4 entradas via **Oráculo MLP** salta para **$84{,}90\%$ (1.265 acertos)**:
- Um ganho de **$+20{,}8$ pontos percentuais** sobre o melhor modelo isolado;
- Apenas **$15{,}10\%$ (225 áudios)** constituem o erro comum inevitável.

In [ ]:
# 3. Gráfico: Desempenho Isolado vs Teto Teórico do Oráculo no MLP em MEDIUM
plt.figure(figsize=(10, 5.5))
colors = ['#95a5a6', '#9b59b6', '#e67e22', '#2980b9', '#27ae60']
bars = plt.bar(df_hits['Representacao'], df_hits['Recall_Pct'], color=colors, width=0.55, edgecolor='black', linewidth=0.8)
bars[-1].set_hatch('//')

plt.axhline(84.90, color='#27ae60', linestyle='--', lw=1.5, alpha=0.7, label='Teto do Oráculo MLP (84.90%)')
plt.axhline(64.09, color='#2980b9', linestyle=':', lw=1.5, alpha=0.7, label='Melhor Isolado MEL (64.09%)')

plt.title('Recall MLP na Classe MEDIUM: Modelos Isolados vs. Teto Teórico do Oráculo', fontweight='bold', pad=12)
plt.ylabel('Recall em MEDIUM (%)')
plt.ylim(0, 100)

for bar in bars:
    h = bar.get_height()
    plt.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom',
                 fontsize=10.5, fontweight='bold', color='#111')

plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 4. Análise de Grau de Consenso em MEDIUM no MLP

A pirâmide de consenso em `MEDIUM` é amplamente superior à de `SMALL`:
- **Unanimidade (4/4 certas): $446$ áudios ($29{,}93\%$)** — quase um terço do total é imune à escolha da transformada;
- **Alto Consenso (3/4 certas):** $282$ áudios ($18{,}93\%$);
- **Consenso Dividido (2/4 certas):** $253$ áudios ($16{,}98\%$);
- **Resgate Exclusivo (1/4 certa):** $284$ áudios ($19{,}06\%$);
- **Erro Comum (0/4 certas):** $225$ áudios ($15{,}10\%$).

### Votação por Consenso em MEDIUM:
- **Unanimidade (4/4):** $29{,}93\%$ ($446$ áudios);
- **Maioria Estrita ($\ge 3/4$):** $48{,}86\%$ ($728$ áudios);
- **Maioria Simples / Empate ($\ge 2/4$):** **$65{,}84\%$ ($981$ áudios)** $\rightarrow$ **Supera o melhor modelo isolado (MEL com $64{,}09\%$) em $+1{,}75$ p.p.!**
- **Oráculo ($\ge 1/4$):** **$84{,}90\%$ ($1.265$ áudios)**.

In [ ]:
# 4. Visualização do Grau de Consenso no MLP
plt.figure(figsize=(9, 5))
labels_cons = ['4 Entradas\n(Unânime)', '3 Entradas\n(Alto Consenso)', '2 Entradas\n(Dividido)', '1 Entrada\n(Exclusivo/Frágil)', '0 Entradas\n(Erro Comum)']
counts_cons = [446, 282, 253, 284, 225]
pcts_cons = [29.93, 18.93, 16.98, 19.06, 15.10]
colors_cons = ['#27ae60', '#2980b9', '#f39c12', '#e74c3c', '#7f8c8d']

bars = plt.bar(labels_cons, counts_cons, color=colors_cons, edgecolor='black', linewidth=0.8, width=0.55)
plt.title('Grau de Consenso MLP em MEDIUM (Total: 1.490 Áudios no 5x2cv)', fontweight='bold', pad=12)
plt.ylabel('Quantidade de Áudios de MEDIUM')
plt.ylim(0, 560)

for bar, pct in zip(bars, pcts_cons):
    h = bar.get_height()
    plt.annotate(f'{h} ({pct:.1f}%)', xy=(bar.get_x() + bar.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom',
                 fontsize=9.5, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Resgates Exclusivos e Padrão de Erro no MLP

### Resgates Exclusivos em MEDIUM:
- **MEL:** Salva $88$ rebocadores ($5{,}91\%$);
- **HHT:** Salva $78$ rebocadores ($5{,}23\%$);
- **DEMON:** Salva $67$ rebocadores ($4{,}50\%$);
- **LOFAR:** Salva $51$ rebocadores ($3{,}42\%$).

### Padrão de Erro Simétrico:
- **MEL:** $56{,}45\%$ dos erros vão para `SMALL` e $36{,}64\%$ para `LARGE`;
- **LOFAR:** $58{,}01\%$ dos erros vão para `SMALL` e $35{,}89\%$ para `LARGE`.
Mais uma vez, confirma-se o forte canal de confusão bidirecional entre lanchas e rebocadores em transformadas baseadas em Fourier.

In [ ]:
# 5. Exibição do Gráfico Oficial Consolidado salvo em disco
from IPython.display import Image, display

img_path = 'reports/analise_complementaridade_mlp_medium_oraculo.png'
if os.path.exists(img_path):
    display(Image(filename=img_path))
else:
    print(f"Imagem {img_path} não encontrada. Execute 'scratch/plot_mlp_complementarity.py' para gerá-la.")

## 6. Comparativo Cruzado: MEDIUM no SVM (Notebook 12) vs. MEDIUM no MLP (Notebook 15)

| Dimensão Analítica | SVM Nyström (Notebook 12) | MLP PyTorch (Notebook 15) | Diagnóstico Científico |
| :--- | :---: | :---: | :--- |
| **Melhor Modelo Isolado** | $60{,}40\%$ (MEL / LOFAR) | **$64{,}09\%$ (MEL)** | O MLP extrai representações contínuas melhores em MEDIUM. |
| **Teto do Oráculo** | **$83{,}69\%$** ($1.247$) | **$84{,}90\%$** ($1.265$) | **Convergência total:** ambas as famílias atingem $\approx 84\text{--}85\%$ de oráculo! |
| **Taxa de Unanimidade (4/4)** | **$27{,}05\%$** ($403$) | **$29{,}93\%$** ($446$) | Quase um terço de consenso unânime em ambas as arquiteturas. |
| **Erro Comum Incorrigível** | $16{,}31\%$ ($243$) | **$15{,}10\%$ ($225$)** | Apenas $15\%$ de perda inevitável. |
| **Votação Simples ($\ge 2/4$)** | $64{,}63\%$ ($+4{,}2$ p.p.) | **$65{,}84\%$ ($+1{,}75$ p.p.)** | A votação por maioria supera os modelos isolados em ambas as famílias! |